# Question 1: Dataset Loading and Structure Analysis

In [5]:
# Import Spark Session
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("Phishing Email Detection") \
    .getOrCreate()

# Reduce Spark warnings
spark.sparkContext.setLogLevel("ERROR")

# Load dataset into first DataFrame
df1 = spark.read.csv(
    "../data/email_security_data.csv",
    header=True,
    inferSchema=True
)

print("="*60)
print("DATASET PREVIEW")
print("="*60)

df1.show(5, truncate=False)

print("\n" + "="*60)
print("DATASET STRUCTURE")
print("="*60)

for col_name, data_type in df1.dtypes:
    print(f"{col_name:<25} {data_type}")

rows = df1.count()
columns = len(df1.columns)

print("\n" + "="*60)
print("DATASET DIMENSIONS")
print("="*60)

print(f"Total Rows    : {rows}")
print(f"Total Columns : {columns}")

DATASET PREVIEW
+----------------+--------+----------------+------------------+---------+-----------+---------------+----------+-------------+
|EmailAttachments|URLCount|SenderReputation|SuspiciousKeywords|EmailSize|LinkEntropy|DomainRiskScore|AccountAge|PhishingLabel|
+----------------+--------+----------------+------------------+---------+-----------+---------------+----------+-------------+
|3               |7       |169             |1                 |259.69   |3.046      |0.239          |78        |1            |
|0               |4       |87              |7                 |322.43   |2.347      |0.494          |24        |0            |
|0               |4       |195             |1                 |92.37    |3.795      |0.583          |11        |0            |
|0               |2       |85              |0                 |447.77   |2.667      |0.123          |72        |0            |
|3               |4       |163             |4                 |294.7    |0.0        |0.889     

# Question 2: Missing Value Replacement using Median

In [11]:
from pyspark.sql.functions import when, col

print("="*60)
print("MEDIAN CALCULATION")
print("="*60)

# Calculate median excluding 0 values

suspicious_median = int(
    df1.filter(
        col("SuspiciousKeywords") != 0
    ).approxQuantile(
        "SuspiciousKeywords",
        [0.5],
        0
    )[0]
)

emailsize_median = float(
    df1.filter(
        col("EmailSize") != 0
    ).approxQuantile(
        "EmailSize",
        [0.5],
        0
    )[0]
)

print(f"Median of SuspiciousKeywords: {suspicious_median}")
print(f"Median of EmailSize: {emailsize_median}")

# Create second DataFrame

df2 = df1.withColumn(
    "SuspiciousKeywords",
    when(
        col("SuspiciousKeywords")==0,
        suspicious_median
    ).otherwise(col("SuspiciousKeywords"))
).withColumn(
    "EmailSize",
    when(
        col("EmailSize")==0,
        emailsize_median
    ).otherwise(col("EmailSize"))
)

print("\n" + "="*60)
print("UPDATED DATA PREVIEW")
print("="*60)

df2.show(10, truncate=False)

MEDIAN CALCULATION
Median of SuspiciousKeywords: 4
Median of EmailSize: 276.12

UPDATED DATA PREVIEW
+----------------+--------+----------------+------------------+---------+-----------+---------------+----------+-------------+
|EmailAttachments|URLCount|SenderReputation|SuspiciousKeywords|EmailSize|LinkEntropy|DomainRiskScore|AccountAge|PhishingLabel|
+----------------+--------+----------------+------------------+---------+-----------+---------------+----------+-------------+
|3               |7       |169             |1                 |259.69   |3.046      |0.239          |78        |1            |
|0               |4       |87              |7                 |322.43   |2.347      |0.494          |24        |0            |
|0               |4       |195             |1                 |92.37    |3.795      |0.583          |11        |0            |
|0               |2       |85              |4                 |447.77   |2.667      |0.123          |72        |0            |
|3        

# Question 3: Dataset Cleaning and Row Removal